# 03 — Modeling (Logistic Regression, SVM, Neural Network)

This notebook trains and evaluates multiple machine learning models on the preprocessed dataset.  
Models included:

- Logistic Regression  
- Support Vector Machines (Linear, RBF, Polynomial, Sigmoid)  
- Neural Network (MLPClassifier)  
- Comparison of all models  

We use the final cleaned dataset from `02_Preprocessing.ipynb`.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load processed dataset
df = pd.read_csv("../data/processed/processed_train.csv")
df.head()

,ApplicantYears,AnnualEarnings,RequestedSum,TrustMetric,WorkDuration,ActiveAccounts,OfferRate,RepayPeriod,DebtFactor,OwnsProperty,...,QualificationLevel_PhD,WorkCategory_Part-time,WorkCategory_Self-employed,WorkCategory_Unemployed,RelationshipStatus_Married,RelationshipStatus_Single,FundUseCase_Business,FundUseCase_Education,FundUseCase_Home,FundUseCase_Other
0,18,137576,209136,846,26,2,10.47,60,0.81,1,...,False,False,True,False,False,True,True,False,False,False
1,47,57194,5970,748,30,2,19.72,36,0.73,0,...,False,False,False,True,False,False,False,True,False,False
2,26,84328,95065,453,7,2,24.25,12,0.45,0,...,False,False,True,False,True,False,False,False,False,True
3,53,49795,229582,533,107,3,14.44,60,0.17,1,...,False,False,True,False,False,True,False,False,False,False
4,49,115450,22072,840,0,4,24.48,12,0.11,0,...,False,True,False,False,False,True,False,True,False,False


### Split data into train and test sets and scale numerical features

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df.drop("RiskFlag", axis=1)
y = df["RiskFlag"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Scale numerical columns only
numerical_cols = [
    "ApplicantYears", "AnnualEarnings", "RequestedSum", "TrustMetric",
    "WorkDuration", "ActiveAccounts", "OfferRate", "RepayPeriod", "DebtFactor"
]

scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

X_train.head()

,ApplicantYears,AnnualEarnings,RequestedSum,TrustMetric,WorkDuration,ActiveAccounts,OfferRate,RepayPeriod,DebtFactor,OwnsProperty,...,QualificationLevel_PhD,WorkCategory_Part-time,WorkCategory_Self-employed,WorkCategory_Unemployed,RelationshipStatus_Married,RelationshipStatus_Single,FundUseCase_Business,FundUseCase_Education,FundUseCase_Home,FundUseCase_Other
10036,0.230581,-1.525060,-0.469351,0.318845,-0.333887,0.445318,1.063223,-0.709094,-0.047400,0,...,False,True,False,False,True,False,False,False,False,True
78194,-0.503236,1.080194,-1.590685,-1.468182,-0.593773,-1.344705,-0.686443,-1.417674,-1.172879,1,...,False,False,False,False,True,False,False,False,False,False
136028,0.697555,-0.282253,-0.257541,-1.594029,0.474648,-0.449693,-1.289777,-0.000513,0.212326,1,...,False,True,False,False,True,False,False,False,False,True
187712,0.897687,-1.355851,-0.533061,-0.901871,1.716325,-1.344705,-1.341060,1.416647,0.472052,0,...,False,False,False,False,False,True,False,False,False,False
63011,-0.703368,-1.651806,-1.719345,-1.373797,-0.882535,0.445318,-1.650268,-0.709094,-1.216167,1,...,False,True,False,False,False,False,False,False,False,True


### Train and evaluate Logistic Regression

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

log_reg = LogisticRegression(max_iter=500)

log_reg.fit(X_train, y_train)

log_preds = log_reg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, log_preds))
print("\nClassification Report:")
print(classification_report(y_test, log_preds))

Accuracy: 0.8849422361464656

Classification Report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     45132
           1       0.59      0.04      0.07      5938

    accuracy                           0.88     51070
   macro avg       0.74      0.52      0.50     51070
weighted avg       0.85      0.88      0.84     51070



### Logistic Regression with Balanced Weights

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

log_reg = LogisticRegression(class_weight = 'balanced',max_iter=500)

log_reg.fit(X_train, y_train)

log_preds = log_reg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, log_preds))
print("\nClassification Report:")
print(classification_report(y_test, log_preds))

Accuracy: 0.6719013119248091

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.67      0.78     45132
           1       0.22      0.70      0.33      5938

    accuracy                           0.67     51070
   macro avg       0.58      0.68      0.56     51070
weighted avg       0.86      0.67      0.73     51070



### RBF SVM using Random Fourier Features (RBFSampler)


In [15]:
from sklearn.kernel_approximation import RBFSampler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Use float32 to reduce RAM
X_train_32 = X_train.astype(np.float32)
X_test_32 = X_test.astype(np.float32)

start = time.time()

# Choose a stable gamma
gamma_value = 1 / X_train_32.shape[1]

print("Using gamma =", gamma_value)

# RBFSampler (RFF)
feature_map_rff = RBFSampler(
    gamma=gamma_value,
    n_components=500,    # bigger than Nystroem allowed
    random_state=42
)

rbf_rff_svm = Pipeline([
    ('rff', feature_map_rff),
    ('svm', LinearSVC(class_weight='balanced', max_iter=5000))
])

rbf_rff_svm.fit(X_train_32, y_train)

rff_preds = rbf_rff_svm.predict(X_test_32)

print("Training Time:", time.time() - start)
print("Accuracy:", accuracy_score(y_test, rff_preds))
print("\nClassification Report:")
print(classification_report(y_test, rff_preds))


Using gamma = 0.041666666666666664
Training Time: 33.77936315536499
Accuracy: 0.683728216173879

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.68      0.79     45132
           1       0.22      0.69      0.34      5938

    accuracy                           0.68     51070
   macro avg       0.58      0.68      0.56     51070
weighted avg       0.86      0.68      0.74     51070



# Faiss code

In [ ]:
import faiss
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.decomposition import PCA
import time

# ----- Prepare data -----
X_train_np = np.ascontiguousarray(np.asarray(X_train).astype(np.float32))
X_test_np  = np.ascontiguousarray(np.asarray(X_test).astype(np.float32))
y_train_arr = np.array(y_train).astype(int)

# ----- PCA -----
PCA_DIM = 16
pca = PCA(n_components=PCA_DIM, random_state=42)
X_train_pca = np.ascontiguousarray(pca.fit_transform(X_train_np).astype(np.float32))
X_test_pca  = np.ascontiguousarray(pca.transform(X_test_np).astype(np.float32))

# Normalize (cosine)
faiss.normalize_L2(X_train_pca)
faiss.normalize_L2(X_test_pca)

# ----- Class weights -----
class_weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train_arr)
w0, w1 = class_weights

# ----- FAISS -----
d = X_train_pca.shape[1]
K = 20                     # bigger K collects minority neighbors
M = 32
index = faiss.IndexHNSWFlat(d, M)
index.hnsw.efConstruction = 200
index.hnsw.efSearch = 1024
index.add(X_train_pca)

# ----- Weighted Probability Computation -----
def get_probabilities():
    distances, neighbors = index.search(X_test_pca, K)
    probs = []

    for dist_list, idx_list in zip(distances, neighbors):
        # cosine sim
        sim = 1 - dist_list / 2
        sim = np.maximum(sim, 1e-6)

        score0 = 0.0
        score1 = 0.0

        # normal weighted voting
        for lbl, s in zip(y_train_arr[idx_list], sim):
            if lbl == 0:
                score0 += s * w0
            else:
                score1 += s * w1

        # softmax-like normalization
        p1 = score1 / (score1 + score0 + 1e-9)
        probs.append(p1)

    return np.array(probs)

start = time.time()
probs = get_probabilities()
print("Computation Time:", time.time() - start)

# ----- Threshold tuning -----
best_f1 = 0
best_thresh = 0.5
y_test_arr = np.array(y_test)

for thresh in np.linspace(0.05, 0.95, 50):
    preds = (probs >= thresh).astype(int)
    f1 = f1_score(y_test_arr, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print("Best Threshold:", best_thresh)
print("Max F1:", best_f1)

# Final predictions
final_preds = (probs >= best_thresh).astype(int)

print("Accuracy:", accuracy_score(y_test_arr, final_preds))
print(classification_report(y_test_arr, final_preds))

Computation Time: 12.243162870407104
Best Threshold: 0.5826530612244898
Max F1: 0.31632169379048825
Accuracy: 0.745819463481496
              precision    recall  f1-score   support

           0       0.92      0.78      0.84     45132
           1       0.23      0.51      0.32      5938

    accuracy                           0.75     51070
   macro avg       0.58      0.64      0.58     51070
weighted avg       0.84      0.75      0.78     51070



: 

### Retrain RBF model with tuned parameters

In [6]:

best_C = 5

best_gamma = 0.06
best_comp = 800

# SAFE number of components for full dataset (avoid kernel crash)
safe_components = min(best_comp, 1000)

print("Final Training with:")
print("gamma =", best_gamma)
print("C =", best_C)
print("n_components =", safe_components)

final_rff = RBFSampler(
    gamma=best_gamma,
    n_components=safe_components,
    random_state=42
)

final_svm = LinearSVC(
    class_weight='balanced',
    C=best_C,
    max_iter=5000,
    random_state=42
)

final_model = Pipeline([
    ('rff', final_rff),
    ('svm', final_svm)
])

final_model.fit(X_train_32, y_train)

final_preds = final_model.predict(X_test_32)

print("Accuracy:", accuracy_score(y_test, final_preds))
print("\nClassification Report:")
print(classification_report(y_test, final_preds))

Final Training with:
gamma = 0.06
C = 5
n_components = 800
Accuracy: 0.6835128255335814

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.68      0.79     45132
           1       0.22      0.69      0.34      5938

    accuracy                           0.68     51070
   macro avg       0.58      0.69      0.56     51070
weighted avg       0.86      0.68      0.74     51070



## Neural Network (MLPClassifier)

In [13]:
from sklearn.neural_network import MLPClassifier
import time
from sklearn.metrics import accuracy_score, classification_report

print("Training baseline NN...")
start = time.time()

nn1 = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    learning_rate="adaptive",
    max_iter=500,
    random_state=42
)

nn1.fit(X_train, y_train)
pred_nn1 = nn1.predict(X_test)

print("Training time:", time.time() - start)
print("Accuracy:", accuracy_score(y_test, pred_nn1))
print(classification_report(y_test, pred_nn1))


Training baseline NN...
Training time: 133.04963088035583
Accuracy: 0.8615625611905228
              precision    recall  f1-score   support

           0       0.90      0.95      0.92     45132
           1       0.31      0.15      0.21      5938

    accuracy                           0.86     51070
   macro avg       0.60      0.55      0.57     51070
weighted avg       0.83      0.86      0.84     51070



### Stable LBGFS Neural Network

In [10]:
import time
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
print("Training Stable-Converging NN (LBFGS)...")
start = time.time()
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

print("Before oversampling:", y_train.value_counts())
print("After oversampling :", y_train_res.value_counts())

nn_stable = MLPClassifier(
    hidden_layer_sizes=(32, 16),   # small & stable
    activation="relu",
    solver="lbfgs",                # guaranteed convergence
    alpha=1e-3,                    # strong regularization
    max_iter=400,                  # enough for lbfgs
    random_state=42
)

nn_stable.fit(X_train_res, y_train_res)   # use oversampled data

pred_nn_stable = nn_stable.predict(X_test)

print("Training time:", time.time() - start)
print("Accuracy:", accuracy_score(y_test, pred_nn_stable))
print("\nClassification Report:")
print(classification_report(y_test, pred_nn_stable))

Training Stable-Converging NN (LBFGS)...
Before oversampling: RiskFlag
0    135392
1     17815
Name: count, dtype: int64
After oversampling : RiskFlag
0    135392
1    135392
Name: count, dtype: int64
Training time: 64.96789574623108
Accuracy: 0.6777168592128451

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.68      0.79     45132
           1       0.22      0.67      0.33      5938

    accuracy                           0.68     51070
   macro avg       0.58      0.68      0.56     51070
weighted avg       0.86      0.68      0.73     51070



/Users/mj/miniconda3/envs/ml_prj2/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 400 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=400).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [13]:
# --- SMOTE + Neural Network + Threshold Tuning ---

import time
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

print("Applying SMOTE...")
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("After SMOTE:\n", y_train_sm.value_counts())

# -------------------------
# Train Neural Network
# -------------------------
print("\nTraining NN on SMOTE data...")
start = time.time()

nn = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),
    activation="relu",
    solver="adam",
    learning_rate="adaptive",
    learning_rate_init=0.001,
    max_iter=400,
    alpha=1e-4,
    random_state=42
)

nn.fit(X_train_sm, y_train_sm)

train_time = time.time() - start
print(f"Training time: {train_time:.2f}s")

# -------------------------
# Threshold tuning
# -------------------------
print("\nRunning threshold tuning...")

probs = nn.predict_proba(X_test)[:, 1]

best_f1 = 0
best_t = 0.5

for t in np.arange(0.05, 0.95, 0.01):
    preds_t = (probs >= t).astype(int)
    f1 = f1_score(y_test, preds_t)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best Threshold:", round(best_t, 3))
print("Best F1-score:", round(best_f1, 4))

# -------------------------
# Final predictions
# -------------------------
final_preds = (probs >= best_t).astype(int)

print("\nFinal Accuracy:", accuracy_score(y_test, final_preds))
print("\nClassification Report:")
print(classification_report(y_test, final_preds))


Applying SMOTE...
Before SMOTE:
 RiskFlag
0    135392
1     17815
Name: count, dtype: int64
After SMOTE:
 RiskFlag
0    135392
1    135392
Name: count, dtype: int64

Training NN on SMOTE data...
Training time: 131.47s

Running threshold tuning...
Best Threshold: 0.49
Best F1-score: 0.3156

Final Accuracy: 0.7468572547483846

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.78      0.84     45132
           1       0.23      0.50      0.32      5938

    accuracy                           0.75     51070
   macro avg       0.58      0.64      0.58     51070
weighted avg       0.84      0.75      0.78     51070

